In [22]:
import pandas as pd
import numpy as np

In [23]:
# 1. LOAD DATASETS
print("Loading datasets...")
main_df = pd.read_csv('academIQ_clean_dataset.csv')
student_assessments = pd.read_csv('studentAssessment.csv')
assessments = pd.read_csv('assessments.csv')

Loading datasets...


In [24]:
# 2. PREPARE ASSESSMENT DATA
# Merge to get the 'due date' for every submission
submissions_with_dates = pd.merge(
    student_assessments,
    assessments[['id_assessment', 'date']],
    on='id_assessment',
    how='left'
)

# Filter out assessments that don't have a fixed deadline
submissions_with_dates = submissions_with_dates.dropna(subset=['date'])

In [25]:
# 3. CALCULATE BEHAVIOR
# Feature 1: Days Early (The "Procrastination" metric)
submissions_with_dates['days_early'] = submissions_with_dates['date'] - submissions_with_dates['date_submitted']

# Feature 2: Is Late Flag (Used to count total late submissions)
submissions_with_dates['is_late'] = (submissions_with_dates['days_early'] < 0).astype(int)

In [31]:
# We condense the history of each student into single numbers
student_features = submissions_with_dates.groupby('id_student').agg({
    'days_early': 'mean',        # Average time before/after deadline
    'is_late': 'sum',            # Total number of late submissions
    'id_assessment': 'count'     # Total number of verified submissions
}).reset_index()

# Rename columns for clarity
student_features.rename(columns={
    'id_student': 'student_id',
    'days_early': 'procrastination_index',
    'is_late': 'late_submission_count',
    'id_assessment': 'verified_submissions_count'
}, inplace=True)

In [32]:
# 5. MERGE WITH MAIN DATASET
final_df = pd.merge(main_df, student_features, on='student_id', how='left')

# 6. HANDLE MISSING VALUES
final_df['late_submission_count'] = final_df['late_submission_count'].fillna(0)

In [38]:
# 7. SAVE
output_filename = 'academIQ_clean_dataset_v2.csv'
final_df.to_csv(output_filename, index=False)

print("Feature Engineering Complete!")
print(f"Features added:'late_submission_count', 'procrastination_index'")
print(f"Saved to: {output_filename}")
print("\nSample Data:")
print(final_df[['student_id', 'late_submission_count']].head())

Feature Engineering Complete!
Features added:'late_submission_count', 'procrastination_index'
Saved to: academIQ_clean_dataset_v2.csv

Sample Data:
   student_id  late_submission_count
0        6516                    0.0
1        8462                    1.0
2        8462                    1.0
3       11391                    0.0
4       23629                    3.0
